# TA DoRA Final Run (A100)

End-to-end Colab notebook for the TA-requested run: caption cache generation, DoRA parameter checks, a small smoke run, final train+val training, and test submission CSV creation.

## 1. Mount Google Drive

In [72]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configure Paths And Run Settings

In [73]:
from pathlib import Path

REPO_URL = "https://github.com/Demetri65/dl-kaggle-competition-final.git"
REPO_REF = "main"
REPO_DIR = Path("/content/dl-kaggle-competition-final")
SOURCE_ENV = Path("/content/.env")
UPLOADER_KEY = "_env_uploader"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/p2p_runs/ta_dora_final_a100"
DRIVE_CAPTION_CACHE_DIR = "/content/drive/MyDrive/p2p_runs/image_captions"
DATA_DIR_OVERRIDE = ""
EXPERIMENT_ID = "ta01_dora_caption_context_512_aug"
CAPCHECK_EXPERIMENT_ID = ""

SMOKE_LIMIT = 16
SMOKE_TRAIN_BATCH_SIZE = 1
SMOKE_GRADIENT_ACCUMULATION = 4
TRAIN_BATCH_SIZE = 16
GRADIENT_ACCUMULATION = 1
EVAL_BATCH_SIZE = 25
INFERENCE_COMPLETION_BATCH_SIZE = 100
CAPTION_BATCH_SIZE = 100
NUM_WORKERS = 4
LOGGING_STEPS = 5
SMOKE_CHECKPOINT_SAVE_STEPS = 5
CHECKPOINT_SAVE_STEPS = 50
CHECKPOINT_MAX_TO_KEEP = 3


## 3. Prepare The Repo And Data

In [74]:
import subprocess
import ipywidgets as widgets
from IPython.display import display


def get_uploaded_file(uploader):
    value = uploader.value
    if isinstance(value, dict):
        filename, uploaded_file = next(iter(value.items()))
        if isinstance(uploaded_file, dict):
            content = uploaded_file.get("content", uploaded_file.get("data"))
        else:
            content = uploaded_file
    else:
        uploaded_file = value[0]
        if isinstance(uploaded_file, dict):
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
        else:
            filename = uploaded_file.name
            content = uploaded_file.content
    payload = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return filename, payload


ready_to_bootstrap = SOURCE_ENV.exists()
if ready_to_bootstrap:
    print(f"Using existing {SOURCE_ENV}")
else:
    uploader = globals().get(UPLOADER_KEY)
    if uploader is None:
        uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
        globals()[UPLOADER_KEY] = uploader
    if not uploader.value:
        display(uploader)
        print("Select your local .env file in the upload widget above, then rerun this cell.")
    else:
        filename, payload = get_uploaded_file(uploader)
        SOURCE_ENV.write_bytes(payload)
        SOURCE_ENV.chmod(0o600)
        print(f"Saved {filename} to {SOURCE_ENV}")
        uploader.close()
        globals().pop(UPLOADER_KEY, None)
        ready_to_bootstrap = True

if ready_to_bootstrap:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    head = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    print(f"Repo synced to {head} from origin/{REPO_REF} at {REPO_DIR}")

    required_repo_files = [
        REPO_DIR / "configs/experiments/stage5_ta_requests/ta01_dora_caption_context_512_aug.yaml",
        REPO_DIR / "configs/experiments/stage5_ta_requests/ta02_rank16_dora_qv_mlp_capcheck.yaml",
        REPO_DIR / "scripts/generate_image_captions.py",
    ]
    missing_repo_files = [str(path.relative_to(REPO_DIR)) for path in required_repo_files if not path.exists()]
    if missing_repo_files:
        raise FileNotFoundError(
            "The Colab checkout does not include the TA DoRA implementation files. "
            f"Missing: {missing_repo_files}. "
            f"Commit and push this work, then set REPO_REF to that branch or commit. Current checkout: {head}."
        )

    config_source = (REPO_DIR / "src/config.py").read_text()
    required_source_markers = ["answer_prefix", "CaptioningConfig", "use_dora", "resume_from_checkpoint"]
    missing_source_markers = [marker for marker in required_source_markers if marker not in config_source]
    if missing_source_markers:
        raise RuntimeError(
            "The Colab checkout has the stage5 config files but not the matching Python implementation. "
            f"Missing source markers in src/config.py: {missing_source_markers}. "
            f"Commit and push the full code changes, then set REPO_REF to that branch or commit. Current checkout: {head}."
        )

    # Colab keeps imported modules alive across cells. After git reset/fetch, clear repo modules
    # so later cells import the freshly synced implementation from disk.
    import importlib
    import sys

    importlib.invalidate_caches()
    for module_name in list(sys.modules):
        if module_name == "src" or module_name.startswith("src.") or module_name == "scripts" or module_name.startswith("scripts."):
            sys.modules.pop(module_name, None)

    result = subprocess.run(
        ["bash", "scripts/bootstrap_colab.sh"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        raise RuntimeError(f"scripts/bootstrap_colab.sh failed with exit code {result.returncode}")

Using existing /content/.env
Repo synced to 16be469 from origin/main at /content/dl-kaggle-competition-final
Using Kaggle CLI Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.1), please consider upgrading to the latest version (2.0.2)
Kaggle CLI 2.0.1
pixels-to-predictions.zip: Skipping, found more recently modified local copy (use --force to force download)
Extracting pixels-to-predictions.zip
Competition files downloaded to: /content/dl-kaggle-competition-final/data


## 4. Helpers And Expected Config

In [75]:
import json
import os
import shutil
import shlex
import subprocess
from pathlib import Path

import yaml

REPO_ROOT = REPO_DIR.resolve()
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_CAPTION_CACHE_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

COMMON_OVERRIDES = [
    f"captioning.cache_dir={DRIVE_CAPTION_CACHE_DIR}",
    f"captioning.batch_size={CAPTION_BATCH_SIZE}",
]
if DATA_DIR_OVERRIDE:
    COMMON_OVERRIDES.append(f"data.data_dir={DATA_DIR_OVERRIDE}")

EXPECTED_FINAL_CONFIG = {
    "experiment_id": EXPERIMENT_ID,
    "parent_experiment_id": "c03_balanced_answer",
    "formulation.mode": "candidate_yes_no",
    "prompting.template": "candidate_yes_no",
    "prompting.answer_prefix": "Verdict:",
    "sampling.mode": "balanced_answer_index",
    "lora.rank": 16,
    "lora.alpha": 32,
    "lora.use_dora": True,
    "lora.target_preset": "attn",
    "captioning.enabled": True,
    "captioning.cache_dir": DRIVE_CAPTION_CACHE_DIR,
    "captioning.batch_size": CAPTION_BATCH_SIZE,
    "image.resize_mode": "aspect_pad",
    "image.target_long_edge": 512,
    "image.augmentation.enabled": True,
    "runtime.final_retrain": True,
    "runtime.predict_test": True,
    "training.checkpointing.enabled": True,
    "training.checkpointing.save_steps": CHECKPOINT_SAVE_STEPS,
    "training.checkpointing.max_to_keep": CHECKPOINT_MAX_TO_KEEP,
    "fields": ["image", "image_caption", "question", "choices", "hint", "lecture", "grade", "subject", "topic"],
}


def with_common_overrides(overrides):
    return [*COMMON_OVERRIDES, *overrides]


def extend_with_overrides(args, overrides):
    for override in overrides:
        args.extend(["--set", override])


def run_repo_command(args, allow_failure=False):
    command = ["python3", *args]
    env = os.environ.copy()
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    print("$", " ".join(shlex.quote(part) for part in command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    try:
        assert process.stdout is not None
        for line in process.stdout:
            output_lines.append(line)
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    output = "".join(output_lines)
    if returncode != 0 and not allow_failure:
        raise RuntimeError(f"Command failed with exit code {returncode}")
    return subprocess.CompletedProcess(command, returncode, stdout=output, stderr="")


def find_run_dir(run_root: Path, experiment_id: str) -> Path:
    candidates = sorted(run_root.glob(f"{experiment_id}_seed*"))
    candidates = [path for path in candidates if (path / "resolved_config.yaml").exists()]
    if not candidates:
        raise FileNotFoundError(f"No resolved_config.yaml found under {run_root}")
    if len(candidates) > 1:
        print(f"Multiple run dirs found; using {candidates[-1]}")
    return candidates[-1]


def find_latest_checkpoint(run_root: Path, experiment_id: str) -> Path | None:
    run_dir = run_root / f"{experiment_id}_seed42"
    checkpoint_root = run_dir / "checkpoints"
    if not checkpoint_root.exists():
        return None
    checkpoints = [
        path for path in checkpoint_root.iterdir()
        if path.is_dir() and (path / "model" / "adapter_config.json").exists()
    ]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda path: (path.stat().st_mtime, path.name))[-1]
    return latest


def selected_fields(config):
    return [name for name, enabled in config["fields"].items() if enabled]


def actual_config_values(config):
    return {
        "experiment_id": config["experiment_id"],
        "parent_experiment_id": config.get("parent_experiment_id"),
        "formulation.mode": config["formulation"]["mode"],
        "prompting.template": config["prompting"]["template"],
        "prompting.answer_prefix": config["prompting"]["answer_prefix"],
        "sampling.mode": config["sampling"]["mode"],
        "lora.rank": config["lora"]["rank"],
        "lora.alpha": config["lora"]["alpha"],
        "lora.use_dora": config["lora"]["use_dora"],
        "lora.target_preset": config["lora"]["target_preset"],
        "captioning.enabled": config["captioning"]["enabled"],
        "captioning.cache_dir": config["captioning"]["cache_dir"],
        "captioning.batch_size": config["captioning"]["batch_size"],
        "image.resize_mode": config["image"]["resize_mode"],
        "image.target_long_edge": config["image"]["target_long_edge"],
        "image.augmentation.enabled": config["image"]["augmentation"]["enabled"],
        "runtime.final_retrain": config["runtime"]["final_retrain"],
        "runtime.predict_test": config["runtime"]["predict_test"],
        "data.data_dir": config["data"]["data_dir"],
        "training.epochs": config["training"]["epochs"],
        "training.bf16": config["training"]["bf16"],
        "training.fp16": config["training"]["fp16"],
        "training.batch_size": config["training"]["batch_size"],
        "training.gradient_accumulation": config["training"]["gradient_accumulation"],
        "training.eval_batch_size": config["training"]["eval_batch_size"],
        "scoring.max_completion_batch_size": config["scoring"]["max_completion_batch_size"],
        "runtime.num_workers": config["runtime"]["num_workers"],
        "runtime.logging_steps": config["runtime"]["logging_steps"],
        "training.checkpointing.enabled": config["training"]["checkpointing"]["enabled"],
        "training.checkpointing.save_steps": config["training"]["checkpointing"]["save_steps"],
        "training.checkpointing.max_to_keep": config["training"]["checkpointing"]["max_to_keep"],
        "fields": selected_fields(config),
    }


def assert_expected_final_config(config):
    actual = actual_config_values(config)
    mismatches = []
    for key, expected in EXPECTED_FINAL_CONFIG.items():
        if actual[key] != expected:
            mismatches.append(f"{key}: expected {expected!r}, got {actual[key]!r}")
    if mismatches:
        raise AssertionError("Resolved config does not match TA DoRA final setup:\n" + "\n".join(mismatches))
    return actual


def print_config_values(values):
    for key, value in values.items():
        print(f"{key}: {value}")


def verify_resolved_config(run_root: Path, experiment_id: str = EXPERIMENT_ID, final: bool = False):
    run_dir = find_run_dir(run_root, experiment_id)
    path = run_dir / "resolved_config.yaml"
    config = yaml.safe_load(path.read_text())
    values = actual_config_values(config)
    if final:
        values = assert_expected_final_config(config)
    print(f"Resolved config: {path}")
    print_config_values(values)
    return run_dir, config


def print_run_artifacts(run_dir: Path):
    summary_path = run_dir / "run_summary.yaml"
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing run summary: {summary_path}")
    summary = yaml.safe_load(summary_path.read_text())
    for key in ("trainable_parameters", "run_summary_path", "resolved_config_path", "val_predictions_path", "test_predictions_path", "submission_path"):
        print(f"{key}: {summary.get(key)}")
    submission_path = summary.get("submission_path")
    if submission_path and not Path(submission_path).exists():
        raise FileNotFoundError(f"Submission was not written: {submission_path}")
    return summary


def recover_local_caption_cache():
    local_cache = REPO_ROOT / "data/cache/image_captions"
    drive_cache = Path(DRIVE_CAPTION_CACHE_DIR)
    drive_cache.mkdir(parents=True, exist_ok=True)

    def line_count(path: Path) -> int:
        return sum(1 for _ in path.open()) if path.exists() else 0

    if not local_cache.exists():
        print(f"No local caption cache found at {local_cache}")
        return
    for path in sorted(local_cache.glob("*.jsonl")):
        target = drive_cache / path.name
        local_rows = line_count(path)
        drive_rows = line_count(target)
        if drive_rows >= local_rows:
            print(f"Keeping Drive {path.name} ({drive_rows} rows); local has {local_rows} rows")
            continue
        shutil.copy2(path, target)
        print(f"Recovered {path.name} to Drive ({local_rows} rows)")


def count_file_lines(path: Path) -> int:
    return sum(1 for _ in path.open()) if path.exists() else 0


def expected_caption_rows(split: str, limit=None) -> int:
    csv_path = REPO_ROOT / "data" / f"{split}.csv"
    if DATA_DIR_OVERRIDE:
        csv_path = Path(DATA_DIR_OVERRIDE) / f"{split}.csv"
        if not csv_path.is_absolute():
            csv_path = REPO_ROOT / csv_path
    total_rows = max(0, count_file_lines(csv_path) - 1)
    return min(total_rows, limit) if limit is not None else total_rows


def caption_cache_is_complete(split: str, limit=None) -> bool:
    cache_path = Path(DRIVE_CAPTION_CACHE_DIR) / f"{split}.jsonl"
    cached_rows = count_file_lines(cache_path)
    expected_rows = expected_caption_rows(split, limit=limit)
    if expected_rows and cached_rows >= expected_rows:
        print(f"Skipping {split}: Drive caption cache already has {cached_rows}/{expected_rows} rows")
        return True
    print(f"Caption cache for {split}: {cached_rows}/{expected_rows} rows; generating missing rows")
    return False


def run_caption_generation_split_by_split(splits, limit=None):
    recover_local_caption_cache()
    for split in splits:
        if caption_cache_is_complete(split, limit=limit):
            continue
        args = [
            "scripts/generate_image_captions.py",
            "--experiment", EXPERIMENT_ID,
            "--splits", split,
        ]
        if limit is not None:
            args.extend(["--limit", str(limit)])
        extend_with_overrides(args, COMMON_OVERRIDES)
        run_repo_command(args)
        recover_local_caption_cache()


recover_local_caption_cache()

print(f"Repo root: {REPO_ROOT}")
print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
print(f"Drive caption cache dir: {DRIVE_CAPTION_CACHE_DIR}")
print(f"Final experiment: {EXPERIMENT_ID}")
print(f"Cap-check experiment: {CAPCHECK_EXPERIMENT_ID}")

Keeping Drive train.jsonl (3109 rows); local has 3109 rows
Keeping Drive val.jsonl (1048 rows); local has 1048 rows
Repo root: /content/dl-kaggle-competition-final
Drive output dir: /content/drive/MyDrive/p2p_runs/ta_dora_final_a100
Drive caption cache dir: /content/drive/MyDrive/p2p_runs/image_captions
Final experiment: ta01_dora_caption_context_512_aug
Cap-check experiment: 


## 5. Preflight Config Check

In [76]:
preflight_overrides = with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    f"training.batch_size={TRAIN_BATCH_SIZE}",
    f"training.gradient_accumulation={GRADIENT_ACCUMULATION}",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.num_workers={NUM_WORKERS}",
    f"runtime.logging_steps={LOGGING_STEPS}",
    "training.checkpointing.enabled=true",
    f"training.checkpointing.save_steps={CHECKPOINT_SAVE_STEPS}",
    f"training.checkpointing.max_to_keep={CHECKPOINT_MAX_TO_KEEP}",
    "runtime.final_retrain=true",
    "runtime.predict_test=true",
])

from src.config import load_experiment_config, selected_fields as resolved_selected_fields

preflight_config = load_experiment_config(
    REPO_ROOT,
    EXPERIMENT_ID,
    cli_overrides=preflight_overrides,
    output_dir=str(Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"),
)
preflight_values = {
    "experiment_id": preflight_config.experiment_id,
    "parent_experiment_id": preflight_config.parent_experiment_id,
    "formulation.mode": preflight_config.formulation.mode,
    "prompting.template": preflight_config.prompting.template,
    "prompting.answer_prefix": preflight_config.prompting.answer_prefix,
    "sampling.mode": preflight_config.sampling.mode,
    "lora.rank": preflight_config.lora.rank,
    "lora.alpha": preflight_config.lora.alpha,
    "lora.use_dora": preflight_config.lora.use_dora,
    "lora.target_preset": preflight_config.lora.target_preset,
    "captioning.enabled": preflight_config.captioning.enabled,
    "captioning.cache_dir": preflight_config.captioning.cache_dir,
    "captioning.batch_size": preflight_config.captioning.batch_size,
    "image.resize_mode": preflight_config.image.resize_mode,
    "image.target_long_edge": preflight_config.image.target_long_edge,
    "image.augmentation.enabled": preflight_config.image.augmentation.enabled,
    "runtime.final_retrain": preflight_config.runtime.final_retrain,
    "runtime.predict_test": preflight_config.runtime.predict_test,
    "runtime.num_workers": preflight_config.runtime.num_workers,
    "training.checkpointing.enabled": preflight_config.training.checkpointing.enabled,
    "training.checkpointing.save_steps": preflight_config.training.checkpointing.save_steps,
    "training.checkpointing.max_to_keep": preflight_config.training.checkpointing.max_to_keep,
    "data.data_dir": preflight_config.data.data_dir,
    "fields": resolved_selected_fields(preflight_config),
}
for key, expected in EXPECTED_FINAL_CONFIG.items():
    actual = preflight_values[key]
    if actual != expected:
        raise AssertionError(f"{key}: expected {expected!r}, got {actual!r}")
print_config_values(preflight_values)

experiment_id: ta01_dora_caption_context_512_aug
parent_experiment_id: c03_balanced_answer
formulation.mode: candidate_yes_no
prompting.template: candidate_yes_no
prompting.answer_prefix: Verdict:
sampling.mode: balanced_answer_index
lora.rank: 16
lora.alpha: 32
lora.use_dora: True
lora.target_preset: attn
captioning.enabled: True
captioning.cache_dir: /content/drive/MyDrive/p2p_runs/image_captions
captioning.batch_size: 100
image.resize_mode: aspect_pad
image.target_long_edge: 512
image.augmentation.enabled: True
runtime.final_retrain: True
runtime.predict_test: True
runtime.num_workers: 4
training.checkpointing.enabled: True
training.checkpointing.save_steps: 50
training.checkpointing.max_to_keep: 3
data.data_dir: data
fields: ['image', 'image_caption', 'question', 'choices', 'hint', 'lecture', 'grade', 'subject', 'topic']


## 6. Generate Small Caption Cache And Smoke Run

In [ ]:
run_caption_generation_split_by_split(["train", "val", "test"], limit=SMOKE_LIMIT)

smoke_root = Path(DRIVE_OUTPUT_DIR) / "smoke"
smoke_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(smoke_root),
]
extend_with_overrides(smoke_args, with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    f"training.batch_size={SMOKE_TRAIN_BATCH_SIZE}",
    f"training.gradient_accumulation={SMOKE_GRADIENT_ACCUMULATION}",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.max_train_examples={SMOKE_LIMIT}",
    f"runtime.max_val_examples={SMOKE_LIMIT}",
    f"runtime.num_workers={NUM_WORKERS}",
    f"runtime.logging_steps={LOGGING_STEPS}",
    "training.checkpointing.enabled=true",
    f"training.checkpointing.save_steps={SMOKE_CHECKPOINT_SAVE_STEPS}",
    f"training.checkpointing.max_to_keep={CHECKPOINT_MAX_TO_KEEP}",
]))
run_repo_command(smoke_args)
smoke_run_dir, _ = verify_resolved_config(smoke_root)
smoke_summary = print_run_artifacts(smoke_run_dir)
if smoke_summary["trainable_parameters"] > 5_000_000:
    raise AssertionError(f"Smoke run exceeded 5M trainable params: {smoke_summary['trainable_parameters']}")

$ python3 scripts/generate_image_captions.py --experiment ta01_dora_caption_context_512_aug --splits train val test --limit 16 --set captioning.batch_size=100


KeyboardInterrupt: 

## 7. Optional MLP Cap Check

MLP targeting is disabled for this final notebook run. The final candidate stays on cap-safe rank-16 attention DoRA via `ta01_dora_caption_context_512_aug`.


In [77]:
if not CAPCHECK_EXPERIMENT_ID:
    print("Skipping MLP cap check. Final notebook is using attention-only rank-16 DoRA via ta01.")
else:
    capcheck_root = Path(DRIVE_OUTPUT_DIR) / "capcheck"
    capcheck_args = [
        "scripts/run_experiment.py",
        "--experiment", CAPCHECK_EXPERIMENT_ID,
        "--output-dir", str(capcheck_root),
    ]
    extend_with_overrides(capcheck_args, with_common_overrides([
        "training.epochs=1",
        "training.bf16=true",
        "training.fp16=false",
        "training.batch_size=1",
        "training.gradient_accumulation=4",
        "runtime.max_train_examples=1",
        "runtime.max_val_examples=0",
        "runtime.num_workers=0",
    ]))
    capcheck_result = run_repo_command(capcheck_args, allow_failure=True)
    if capcheck_result.returncode == 0:
        capcheck_run_dir, _ = verify_resolved_config(capcheck_root, experiment_id=CAPCHECK_EXPERIMENT_ID)
        capcheck_summary = print_run_artifacts(capcheck_run_dir)
        print(f"Rank-16 q/v+MLP DoRA trainable parameters: {capcheck_summary['trainable_parameters']}")
    else:
        if "exceeds trainable parameter cap" in capcheck_result.stdout:
            print("Rank-16 q/v+MLP DoRA exceeds the 5M trainable-parameter cap. Continuing with cap-safe ta01.")
        else:
            raise RuntimeError("Cap-check run failed for a reason other than the parameter cap.")


Skipping MLP cap check. Final notebook is using attention-only rank-16 DoRA via ta01.


## 8. Generate Full Caption Cache

In [ ]:
run_caption_generation_split_by_split(["train", "val", "test"])

Keeping Drive train.jsonl (3109 rows); local has 3109 rows
Keeping Drive val.jsonl (1048 rows); local has 1048 rows
$ python3 scripts/generate_image_captions.py --experiment ta01_dora_caption_context_512_aug --splits train --set captioning.cache_dir=/content/drive/MyDrive/p2p_runs/image_captions --set captioning.batch_size=100
2026-05-07 02:36:11.553347: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-07 02:36:11.605839: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate c

## 9. Final Train+Val Test Prediction

In [78]:
final_root = Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"
final_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(final_root),
    "--final-retrain",
    "--predict-test",
]
final_overrides = with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    f"training.batch_size={TRAIN_BATCH_SIZE}",
    f"training.gradient_accumulation={GRADIENT_ACCUMULATION}",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.num_workers={NUM_WORKERS}",
    f"runtime.logging_steps={LOGGING_STEPS}",
    "training.checkpointing.enabled=true",
    f"training.checkpointing.save_steps={CHECKPOINT_SAVE_STEPS}",
    f"training.checkpointing.max_to_keep={CHECKPOINT_MAX_TO_KEEP}",
])
resume_checkpoint = find_latest_checkpoint(final_root, EXPERIMENT_ID)
if resume_checkpoint is not None:
    print(f"Resuming final training from latest checkpoint: {resume_checkpoint}")
    final_overrides.append(f"runtime.resume_from_checkpoint={resume_checkpoint}")
extend_with_overrides(final_args, final_overrides)
run_repo_command(final_args)
final_run_dir, resolved_config = verify_resolved_config(final_root, final=True)
final_summary = print_run_artifacts(final_run_dir)
if final_summary["trainable_parameters"] > 5_000_000:
    raise AssertionError(f"Final run exceeded 5M trainable params: {final_summary['trainable_parameters']}")
if not final_summary.get("latest_checkpoint_dir"):
    raise AssertionError("Final run did not record a latest checkpoint directory")
print("Latest checkpoint artifact:")
print(Path(final_summary["latest_checkpoint_dir"]))
print("Final submission artifact:")
print(Path(final_summary["submission_path"]))

Resuming final training from latest checkpoint: /content/drive/MyDrive/p2p_runs/ta_dora_final_a100/final_train_val_test/ta01_dora_caption_context_512_aug_seed42/checkpoints/step_epoch001_step000100
$ python3 scripts/run_experiment.py --experiment ta01_dora_caption_context_512_aug --output-dir /content/drive/MyDrive/p2p_runs/ta_dora_final_a100/final_train_val_test --final-retrain --predict-test --set captioning.cache_dir=/content/drive/MyDrive/p2p_runs/image_captions --set captioning.batch_size=100 --set training.epochs=1 --set training.bf16=true --set training.fp16=false --set training.batch_size=16 --set training.gradient_accumulation=1 --set training.eval_batch_size=25 --set scoring.max_completion_batch_size=100 --set runtime.num_workers=4 --set runtime.logging_steps=5 --set training.checkpointing.enabled=true --set training.checkpointing.save_steps=50 --set training.checkpointing.max_to_keep=3 --set runtime.resume_from_checkpoint=/content/drive/MyDrive/p2p_runs/ta_dora_final_a100/fi

RuntimeError: Command failed with exit code 1

## 10. Resume Final Training Safely After OOM

In [ ]:
safe_final_root = Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"
safe_resume_checkpoint = find_latest_checkpoint(safe_final_root, EXPERIMENT_ID)
if safe_resume_checkpoint is None:
    raise FileNotFoundError(f"No checkpoint found under {safe_final_root}")

print(f"Safely resuming final training from latest checkpoint: {safe_resume_checkpoint}")
safe_resume_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(safe_final_root),
    "--final-retrain",
    "--predict-test",
]
safe_resume_overrides = with_common_overrides([
    "training.epochs=1",
    "training.bf16=true",
    "training.fp16=false",
    "training.batch_size=8",
    "training.gradient_accumulation=1",
    "training.eval_batch_size=16",
    "scoring.max_completion_batch_size=64",
    f"runtime.num_workers={NUM_WORKERS}",
    f"runtime.logging_steps={LOGGING_STEPS}",
    "training.checkpointing.enabled=true",
    "training.checkpointing.save_steps=50",
    "training.checkpointing.max_to_keep=1000",
    f"runtime.resume_from_checkpoint={safe_resume_checkpoint}",
])
extend_with_overrides(safe_resume_args, safe_resume_overrides)
run_repo_command(safe_resume_args)
final_run_dir, resolved_config = verify_resolved_config(safe_final_root, final=True)
final_summary = print_run_artifacts(final_run_dir)
print("Latest checkpoint artifact:")
print(Path(final_summary["latest_checkpoint_dir"]))
print("Final submission artifact:")
print(Path(final_summary["submission_path"]))

Safely resuming final training from latest checkpoint: /content/drive/MyDrive/p2p_runs/ta_dora_final_a100/final_train_val_test/ta01_dora_caption_context_512_aug_seed42/checkpoints/epoch_epoch001_step000807
$ python3 scripts/run_experiment.py --experiment ta01_dora_caption_context_512_aug --output-dir /content/drive/MyDrive/p2p_runs/ta_dora_final_a100/final_train_val_test --final-retrain --predict-test --set captioning.cache_dir=/content/drive/MyDrive/p2p_runs/image_captions --set captioning.batch_size=100 --set training.epochs=1 --set training.bf16=true --set training.fp16=false --set training.batch_size=8 --set training.gradient_accumulation=1 --set training.eval_batch_size=16 --set scoring.max_completion_batch_size=64 --set runtime.num_workers=4 --set runtime.logging_steps=5 --set training.checkpointing.enabled=true --set training.checkpointing.save_steps=50 --set training.checkpointing.max_to_keep=1000 --set runtime.resume_from_checkpoint=/content/drive/MyDrive/p2p_runs/ta_dora_fina

## 11. Run Test Inference From Latest Checkpoint

In [84]:
checkpoint_eval_root = Path(DRIVE_OUTPUT_DIR) / "checkpoint_test_eval"
checkpoint_source_root = Path(DRIVE_OUTPUT_DIR) / "final_train_val_test"
latest_checkpoint = find_latest_checkpoint(checkpoint_source_root, EXPERIMENT_ID)
if latest_checkpoint is None:
    raise FileNotFoundError(f"No checkpoint found under {checkpoint_source_root}")

patched_checkpoint = checkpoint_eval_root / "patched_eval_artifact" / latest_checkpoint.name
if patched_checkpoint.exists():
    shutil.rmtree(patched_checkpoint)
patched_checkpoint.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(latest_checkpoint, patched_checkpoint)

patched_config_path = patched_checkpoint / "resolved_config.yaml"
patched_config = yaml.safe_load(patched_config_path.read_text())
patched_config.setdefault("runtime", {})["resume_from_checkpoint"] = None
patched_config["runtime"]["eval_artifact_dir"] = None
patched_config["runtime"]["final_retrain"] = False
with patched_config_path.open("w") as handle:
    yaml.safe_dump(patched_config, handle, sort_keys=False)

print("Using patched checkpoint for inference:", patched_checkpoint)

eval_args = [
    "scripts/run_experiment.py",
    "--experiment", EXPERIMENT_ID,
    "--output-dir", str(checkpoint_eval_root),
    "--predict-test",
]
extend_with_overrides(eval_args, with_common_overrides([
    "training.epochs=0",
    "training.bf16=true",
    "training.fp16=false",
    f"runtime.eval_artifact_dir={patched_checkpoint}",
    "runtime.predict_test=true",
    "runtime.max_val_examples=0",
    "training.eval_batch_size=25",
    "scoring.max_completion_batch_size=100",
    f"runtime.num_workers={NUM_WORKERS}",
]))
run_repo_command(eval_args)

eval_run_dir, _ = verify_resolved_config(checkpoint_eval_root, final=False)
final_summary = print_run_artifacts(eval_run_dir)
print("Submission artifact:")
print(Path(final_summary["submission_path"]))


Using patched checkpoint for inference: /content/drive/MyDrive/p2p_runs/ta_dora_final_a100/checkpoint_test_eval/patched_eval_artifact/step_epoch001_step000550
$ python3 scripts/run_experiment.py --experiment ta01_dora_caption_context_512_aug --output-dir /content/drive/MyDrive/p2p_runs/ta_dora_final_a100/checkpoint_test_eval --predict-test --set captioning.cache_dir=/content/drive/MyDrive/p2p_runs/image_captions --set captioning.batch_size=100 --set training.epochs=0 --set training.bf16=true --set training.fp16=false --set runtime.eval_artifact_dir=/content/drive/MyDrive/p2p_runs/ta_dora_final_a100/checkpoint_test_eval/patched_eval_artifact/step_epoch001_step000550 --set runtime.predict_test=true --set runtime.max_val_examples=0 --set training.eval_batch_size=25 --set scoring.max_completion_batch_size=100 --set runtime.num_workers=4
2026-05-07 04:31:57.257880: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to fl

## 12. Validate Submission CSV

In [ ]:
import pandas as pd

submission_path = Path(final_summary["submission_path"])
submission = pd.read_csv(submission_path)
expected_columns = ["id", "answer"]
if list(submission.columns) != expected_columns:
    raise AssertionError(f"Expected columns {expected_columns}, got {list(submission.columns)}")
if submission.empty:
    raise AssertionError("Submission CSV is empty")
if not pd.api.types.is_integer_dtype(submission["answer"]):
    raise AssertionError("Submission answer column must contain integer choice indices")
print(submission.head())
print(f"Submission rows: {len(submission)}")
print(f"Submission path: {submission_path}")

## 13. Summarize Recorded Results

In [ ]:
summary_args = [
    "scripts/summarize_results.py",
    "--sort-by", "timestamp",
    "--top", "20",
]
run_repo_command(summary_args)